## Purpose of This Notebook

This notebook serves as an exploratory tool for examining the log files produced during the horse behavioural experiments conducted in October and November 2023. It facilitates the experimentation with text parsing techniques on the files before they are imported into a database. The primary objectives are:


> To conduct experiments with regular expressions ([`regex`](https://docs.python.org/3/howto/regex.html)) aimed at extracting pertinent data and fields from the log files.

In [ ]:
# | echo: false

import inspect
import sys
from dataclasses import fields
from pathlib import Path
from pprint import pprint

import duckdb
import pandas as pd
from horse_logic.logfiles import Event, Experiment, Logfile, Logs, Parameter, Trial
from IPython.display import Markdown, display
from loguru import logger

In [ ]:
# | echo: false

def display_class_definition(class_name):
    display(Markdown(f"```{inspect.getsource(class_name)}```"))

In [ ]:
# | echo: false

import itables.options as opt
from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)  # Display dataframes in a more friendly paginated manner
opt.pageLength = 20


In [ ]:
# | echo: false

# Custom notebook functions


def export_data_to_csv(df, info_message=None, filepath=None, header=None):
    df.to_csv(filepath, header=header)
    logger.info(f"{len(df)} rows - {info_message}: {filepath}")

def set_custom_logger_format():
    # custom_format = "<level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>"
    custom_format_no_func_info = "<level>{level: <8}</level> | <level>{message}</level>"
    logger.remove()    # Remove the default handler before adding a new one
    logger.add(sys.stderr, format=custom_format_no_func_info)

set_custom_logger_format()

In [ ]:
# | echo: false

# Custom notebook analysis functions

def display_sessions_by_subject(df_out, subject_df):
    session_summary = []
    for n_subject, subject_name in enumerate(subject_df["subject_name"]):
        df_ = df_out.drop(columns=["suffix", "subject_name"])[df_out["subject_name"] == subject_name]
        n_session = len(df_)
        session_summary.append((n_subject+1, subject_name.capitalize(), n_session))
        if n_session == 0:
            logger.info(f"Subject {subject_name}: No experiments conducted")
        else:
            df_.loc[:, "time_dff"] = df_["datetime"].diff()
            display(Markdown(f"### {n_subject+1}. {subject_name.capitalize()}: {n_session} session(s)"))
            display(df_)
    return pd.DataFrame(session_summary, columns=["Subject number", "Subject name", "Session count"])

### Load Subject information

In [ ]:
# | echo: false

HORSE_ORDER_XLSX = "Horseorder ALL exp 1.xlsx"
HORSE_ORDER_PATH = Path("../docs/provided") 

HORSE_ORDER_FILEPATH = HORSE_ORDER_PATH / HORSE_ORDER_XLSX

if (HORSE_ORDER_FILEPATH).exists():
    subject_df = pd.read_excel(HORSE_ORDER_FILEPATH)
    logger.info(f"Loaded horse order info from: {HORSE_ORDER_FILEPATH}")
else:
    logger.error(f"Horse order info not found: {HORSE_ORDER_FILEPATH}")

In [ ]:
# | echo: false

subject_df.rename({"No": "subject_number", "Horse": "subject_name"}, axis=1, inplace=True)
subject_df["subject_name"] = subject_df["subject_name"].str.lower()     # ensure lower case names for later lookup


## Log file reconciliation

### Directory information: log file data and outputs

In [ ]:
# | echo: false

LOGFILES_DIR = "../data/results/zips/cb_data"

assert Path(LOGFILES_DIR).exists()
logger.info(f"Logfiles dir: {LOGFILES_DIR}")


In [ ]:
# | echo: false

OUTPUT_DIR = "../data/results/notebooks/cb"

assert Path(OUTPUT_DIR).exists()
logger.info(f"Outputs dir: {OUTPUT_DIR}")

In [ ]:
# | echo: false

DATA_DIR = "../data"

assert Path(DATA_DIR).exists()

DATA_DB  = Path(DATA_DIR) / "experiments_October2023-cb.ddb"  # DuckDB name
logger.info(f"Database file: {DATA_DB}")

### Load logfiles from CSV list of included files

As determined in `logfiles-reconciliation-cb.ipynb`

In [ ]:
# | echo: false

logs = Logs(path=LOGFILES_DIR)
logs.load_specific_logfiles_from_csv(Path(OUTPUT_DIR) / "all_included_files.csv")

In [ ]:
# | echo: false

logger.info(logs)

### Explore `regex` parsing of a single example log file

Analysis code is in `logfiles.py`. Explore parsing log information for specific example defined below.

In [ ]:
log_eg = [log.file_name for log in logs.logfiles][0]

In [ ]:
# | echo: false

logfile_eg = f"{logs.path}/{log_eg}"

In [ ]:
# | echo: false

# Load example log file into Logfile class for processing
log_eg = Logfile(logfile_eg)

In [ ]:
# | echo: false

print(f"Example log file for regex parsing: {log_eg.file_name}")

### Data structures

These are created as `dataclasses` in Python (in `logfiles.py`) and TABLES in SQL (`sql/create_experiment_tables_ddb.sql`).

#### Experiment table

In [ ]:
# | echo: false

display_class_definition(Experiment)

#### e.g. Parse experiment details and comment from log file

In [ ]:
# | echo: false

pprint(log_eg.parse_filename_components_cog_bias())

In [ ]:
# | echo: false

log_eg.parse_comments()

#### Parameter table

Note: These still need to be added to the database (skipped for now).

In [ ]:
# | echo: false

display_class_definition(Parameter)

##### e.g. Parse Parameters from log file

In [ ]:
# | echo: false

log_eg.parse_parameters()

#### Trials and Events tables

In [ ]:
# | echo: false

display_class_definition(Trial)

In [ ]:
# | echo: false

display_class_definition(Event)

#### e.g. Parsing of trials and events info from log file

In [ ]:
# | echo: false

n_trial = len(log_eg.parse_trials_and_events_cog_bias()[0])

In [ ]:
n_trial

In [ ]:
# | echo: false
 
df_trial_event = pd.DataFrame(log_eg.parse_trials_and_events_cog_bias()[1])

In [ ]:
# | echo: false

df_trial_event

##### Example of Trials and Event information

Example of Events for each trial number

In [ ]:
# | echo: false

def display_events_by_trial_number(df_trial_event):
    n_trial = df_trial_event["TrialNumber"].max()
    for i_trial in range(n_trial):
        df_trial_event.loc[:, "time_diff"] = pd.to_datetime(df_trial_event["EventDateTime"]).diff()
        display(Markdown(f"### Trial #{i_trial+1}"))
        display(df_trial_event[df_trial_event["TrialNumber"] == i_trial+1])

display_events_by_trial_number(df_trial_event)

### Bringing this all together and putting the data in DuckDB database

In [ ]:
# | echo: false

def insert_record_to_database(con, record):
    table_name = record.__class__.__name__.lower() + 's'  # Assumes table name is plural of the class name
    columns = ', '.join(field.name for field in fields(record))
    placeholders = ', '.join('?' for _ in fields(record))
    values = tuple(getattr(record, field.name) for field in fields(record))
    query = f'INSERT INTO {table_name} ({columns}) VALUES ({placeholders})'
    # try:
    con.execute(query, values)
    # except Exception as e:
    #     print(f"Query: {query}")
    #     print(f"Values: {values}")

In [ ]:
# | echo: false

if DATA_DB.exists():  # remove database file if it exists
    logger.info(f"Deleted existing database file: {DATA_DB}")
    DATA_DB.unlink()
    
con = duckdb.connect(database=str(DATA_DB))

In [ ]:
# | echo: false

def create_tables_from_sql_file(con, file_path, echo_sql=False):
    with open(file_path, 'r') as file:
        sql_commands = file.read()
    if echo_sql:
        display(Markdown(f"\n```\n{sql_commands}\n```"))
    con.execute(sql_commands)

In [ ]:
# | echo: false

create_tables_from_sql_file(con, '../sql/create_experiment_tables_ddb.sql')

In [ ]:
# | echo: false

# Used during testing

# def load_parse_experiment_log(logfile, path="../data/results/zips/data_17Jan2020_email_hillydale_equine/"):
#      return Logfile(f"{path}/{logfile}").parse_filename_components(), pd.DataFrame(Logfile(f"{path}/{logfile}").parse_trials_and_events()[1])


# exp_details, trial_details = load_parse_experiment_log("Experiment_2023-10-02T21:28:43.621970_bonnie_2_RPE-R.log")

# trial_details

In [ ]:
# | echo: false

extra_excludes = sorted([
    "Experiment_2023-10-05T12:35:54.059431_bonnie_42_Training - fixed.log",
    "Experiment_2023-10-05T12:41:54.062043_bonnie_43_Training randomised Type 1.log",
    "Experiment_2023-10-05T13:48:28.145688_bonnie_45_Training - fixed.log"
])

In [ ]:
# | echo: false

export_data_to_csv(pd.DataFrame(extra_excludes),
                   "Additional log files excluded as bad data",
                   f"{Path(OUTPUT_DIR) / 'extra_excludes.csv'}")

In [ ]:
# | echo: false

logfiles_to_process = [logfile.file_name for logfile in logs.logfiles if logfile.file_name not in extra_excludes]

export_data_to_csv(pd.DataFrame(logfiles_to_process),
                   "List of logfiles to be loaded to database",
                   f"{Path(OUTPUT_DIR) / 'logfiles-to-database.csv'}")

In [ ]:
con

In [ ]:
# | echo: false

def parse_and_load_logfiles_to_database(logfiles_to_process, con):

    # Initialise the unique (integer) id for each Trial (TrialID) and each Event (EventID)
    trial_id = 0
    event_id = 0

    # Loop over Experiments

    for exp_id, logfile in enumerate(sorted(logfiles_to_process)): # Loop over Experiments - starting at exp_id = 0 (ExperimentID)
        logger.info(f"{exp_id}: {logs.path}/{logfile}")
        logf = Logfile(f"{logs.path}/{logfile}")
        try:
            exp_details = logf.parse_filename_components()
            exp_details.ExperimentID = exp_id
            exp_details.SubjectNumber = int(subject_df.loc[subject_df["subject_name"] == exp_details.SubjectName, "subject_number"].iloc[0])   #logger.info(f"Experiment: {exp_details}")

            insert_record_to_database(con, exp_details) # Add each Experiment

            # Loop over Trials
            
            trial_events_df = pd.DataFrame(logf.parse_trials_and_events()[1])
            n_trial = trial_events_df["TrialNumber"].max()

            for trial_number in range(n_trial):
                trial_events_df.loc[:, "EventDateTime"] = pd.to_datetime(trial_events_df["EventDateTime"])
                trial_events_df.loc[:, "time_diff"] = trial_events_df["EventDateTime"].diff()

                trial_start_time = trial_events_df.iloc[0]["EventDateTime"]   # Start tone of the first event in the trial
                trial_end_time = trial_events_df.iloc[-1]["EventDateTime"]    # Last event time in the trial

                insert_record_to_database(con, Trial(trial_id, exp_id, trial_number+1, trial_start_time, trial_end_time)) # Add each Trial
                
                # Loop over Events

                events_df = trial_events_df[trial_events_df["TrialNumber"] == trial_number+1]
                n_event = len(events_df)

                for event_number in range(n_event):

                    event_time = events_df.iloc[event_number]["EventDateTime"]
                    event_type = events_df.iloc[event_number]["EventType"]
                    event_elapsed_time = (event_time - trial_start_time).seconds
                    insert_record_to_database(con, Event(event_id, trial_id, event_number+1, event_time, event_type, event_elapsed_time))   # Add each Event

                    event_id += 1
                trial_id += 1
        except Exception as e:
            print(e)

    return len(logfiles_to_process), trial_id, event_id


In [ ]:
# | echo: false

n_experiment, n_trial, n_event = parse_and_load_logfiles_to_database(logfiles_to_process, con)

In [ ]:
# | echo: false

logger.info(f"# Loaded - Experiments: {n_experiment}, Trials: {n_trial}, Events: {n_event}")

In [ ]:
# Cross-check queries

con.sql("SHOW TABLES;")

In [ ]:
experiments_df = con.sql("SELECT * FROM Experiments").df()

In [ ]:
# | echo: false

experiments_df

In [ ]:
con.sql("SELECT DISTINCT ExperimentType FROM Experiments")

In [ ]:
con.sql("SELECT DISTINCT SubjectName FROM Experiments ORDER BY SubjectName")

In [ ]:
con.sql("SELECT * FROM Trials")

In [ ]:
con.sql("SELECT DISTINCT TrialStartTime FROM Trials")

In [ ]:
events_ddb = con.sql("SELECT * FROM Events").df()

In [ ]:
# | echo: false 

events_ddb.head(10)

In [ ]:
# | echo: false

events_ddb.tail(10)

In [ ]:
event_type_ddb = con.sql("SELECT DISTINCT EventType FROM Events ORDER BY EventType").df()

In [ ]:
# | echo: false

event_type_ddb

In [ ]:
con.close()